# Learn the Massachusetts parser
A saved official PDF ? raw text ? the module parser ? a DataFrame ? reconciliation.
This notebook runs offline and does not write files or SQLite.

Category 1 is retail; Category 3 is online/mobile. We inspect Category 3 only.
`Total Online` is a control total, not an operator.
The published Accrual Win and Taxable Gaming Revenue are separate measures.

Source: [Massachusetts Gaming Commission](https://massgaming.com/regulations/revenue/).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import pdfplumber

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.states import massachusetts as ma

pdf_path = ROOT / "tests" / "fixtures" / "MA" / "MGC-Revenue-Report-July-2026.pdf"
print(pdf_path)

## 1. Inspect the saved report before parsing
The source prints settled wagers, Accrual Win, Hold %, taxable revenue, and tax.
The excerpt below makes the source layout visible.

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    online_text = ma.find_online_operator_text(pdf)
print(online_text[:1800])

## 2. Call the same parser used by collection
`parse_revenue_pdf` checks the report period, finds online operators, and reconciles
all four money fields to Total Online. There is no second notebook-local parser.

In [ ]:
operators, (year, month) = ma.parse_revenue_pdf(pdf_path.read_bytes())
parsed = pd.DataFrame(operators)
print(f"Reporting month: {year}-{month:02d}")
display(parsed)

## 3. Show the reconciliation
The module defines the tolerances because the printed totals may have rounding differences.
Dropping an operator would cause this check to fail.

In [ ]:
_, total_online = ma.parse_online_section(online_text)
ma.reconcile_operators(operators, total_online)
fields = ["wagers_settled", "accrual_win", "taxable_revenue", "tax_collected"]
check = pd.DataFrame({"operator_sum": parsed[fields].sum(), "printed_total": pd.Series(total_online)})
check["difference"] = check["operator_sum"] - check["printed_total"]
check["allowed_difference"] = pd.Series(ma.RECONCILE_TOLERANCE)
display(check)
print("The module reconciliation passed.")

## 4. Understand the stored columns
| Source column | SQLite column |
| --- | --- |
| Current & Future Wagers Settled | `handle` |
| Accrual Win by Licensee | `gross_revenue` |
| Taxable Gaming Revenue | `taxable_revenue` |
| Tax Collected | `tax` |

Accrual Win is already retained by the current parser. It does not replace taxable revenue.
This example proves the reconciliation for this file; it does not certify every historical row
or make Massachusetts economically identical to other states.

To study another layout, change `pdf_path` to `March-Rev-Report.pdf` in the same folder.
Use notebook 20 for collection and notebook 90 for analysis.